# Evaluate Elo of a PyTorch Chess Engine using Cutechess-cli

This notebook demonstrates how to load a PyTorch model checkpoint, wrap it as a basic UCI (Universal Chess Interface) chess engine, and then evaluate its Elo rating against a known engine (e.g., Stockfish) using `cutechess-cli`.

In [ ]:
import torch
import torch.nn as nn
import os
import subprocess
import sys
import time

## 1. Define the Model Architecture
**IMPORTANT**: This is a placeholder model. You must replace `SimpleChessNet` with the actual PyTorch model definition used in your `train.py` script. The architecture must exactly match what was used to create the checkpoint.

In [ ]:
# Placeholder for your actual model architecture
class SimpleChessNet(nn.Module):
    def __init__(self):
        super(SimpleChessNet, self).__init__()
        # Example layers - replace with your actual model layers
        self.fc1 = nn.Linear(768, 256) # Assuming some input feature size
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)  # Outputting a single evaluation score

    def forward(self, x):
        # Example forward pass - replace with your actual model's forward pass
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Define the path to your PyTorch checkpoint file
CHECKPOINT_PATH = '/data/trained_nets/your_model_epoch_X.pth' # Update this path

## 2. Load the PyTorch Checkpoint
We will load the trained model from the specified checkpoint path.

In [ ]:
try:
    model = SimpleChessNet() # Instantiate your model
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=torch.device('cpu')))
    model.eval() # Set the model to evaluation mode
    print(f"Successfully loaded model from {CHECKPOINT_PATH}")
except FileNotFoundError:
    print(f"Error: Checkpoint file not found at {CHECKPOINT_PATH}")
    sys.exit(1)
except Exception as e:
    print(f"Error loading model: {e}")
    sys.exit(1)

## 3. Create a UCI Engine Wrapper Script
To interact with `cutechess-cli`, our PyTorch model needs to be wrapped inside a script that understands the UCI protocol. This script will load the trained PyTorch model and use it to make moves. 

**Note**: A full UCI engine implementation is complex and beyond the scope of a simple notebook cell. This section provides a conceptual outline and a placeholder for such a script. You would typically develop a separate Python file (`my_pytorch_engine.py`) that implements the UCI protocol and integrates your `model` for move generation/evaluation.

In [ ]:
UCI_ENGINE_SCRIPT_NAME = 'my_pytorch_engine.py'

uci_engine_content = f"""
import torch
import torch.nn as nn
import sys
import chess
import chess.uci
import random # For placeholder move generation

# IMPORTANT: This model definition MUST match the one used during training
class SimpleChessNet(nn.Module):
    def __init__(self):
        super(SimpleChessNet, self).__init__()
        self.fc1 = nn.Linear(768, 256) # Example size, adjust as per your model
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

CHECKPOINT_PATH = '{CHECKPOINT_PATH}' # Path to the PyTorch checkpoint

def load_model():
    model = SimpleChessNet()
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=torch.device('cpu')))
    model.eval()
    return model

def main():
    board = chess.Board()
    model = None
    
    while True:
        line = sys.stdin.readline().strip()
        if line == 'uci':
            print('id name MyPyTorchEngine')
            print('id author Gemini Code Assist')
            print('uciok')
        elif line == 'isready':
            if model is None:
                model = load_model()
            print('readyok')
        elif line == 'ucinewgame':
            board.reset()
        elif line.startswith('position'):
            parts = line.split(' ')
            if 'startpos' in parts:
                board.reset()
                move_index = parts.index('startpos') + 1
            else:
                # Handle FEN if necessary
                move_index = -1 # Simplified, assume startpos for now
            
            if 'moves' in parts:
                moves_start_index = parts.index('moves') + 1
                for i in range(moves_start_index, len(parts)):
                    move = chess.Move.from_uci(parts[i])
                    board.push(move)
        elif line.startswith('go'):
            # This is where your PyTorch model would be used to select a move.
            # For demonstration, we'll just pick a random legal move.
            # In a real engine, you'd convert board to model input, run model, interpret output.
            legal_moves = list(board.legal_moves)
            if legal_moves:
                best_move = random.choice(legal_moves)
                print(f'bestmove {best_move.uci()}')
            else:
                print('bestmove (none)') # No legal moves
        elif line == 'quit':
            break

if __name__ == '__main__':
    main()
""".format(CHECKPOINT_PATH=CHECKPOINT_PATH)

with open(UCI_ENGINE_SCRIPT_NAME, 'w') as f:
    f.write(uci_engine_content)

print(f"UCI engine wrapper script '{UCI_ENGINE_SCRIPT_NAME}' created.")

## 4. Evaluate Elo using Cutechess-cli
Now we can use `cutechess-cli` to run a tournament and estimate the Elo rating of our engine. You'll need `cutechess-cli` and an opponent engine (like Stockfish) installed and accessible in your system's PATH.

**Before running:**
- Ensure `cutechess-cli` is installed and in your PATH.
- Ensure Stockfish (or your chosen opponent engine) is installed and its executable is in your PATH or specified correctly.
- Adjust the `OPPONENT_ENGINE_PATH` and tournament parameters as needed.

In [ ]:
OPPONENT_ENGINE_PATH = 'stockfish' # Or full path, e.g., '/usr/local/bin/stockfish'
RESULTS_FILE = 'elo_evaluation_results.txt'

cutechess_command = [
    'cutechess-cli',
    '-engine', f'cmd=python {UCI_ENGINE_SCRIPT_NAME}', 'name=MyPyTorchEngine',
    '-engine', f'cmd={OPPONENT_ENGINE_PATH}', 'name=Stockfish',
    '-each', 'proto=uci', 'tc=10/0.1', 'depth=10', # Time control and search depth for each engine
    '-rounds', '2', # Number of rounds (each engine plays white and black once)
    '-games', '20', # Total number of games (rounds * 2 for white/black)
    '-ratinginterval', '10', # Update ratings every 10 games
    '-pgnout', RESULTS_FILE,
    '-concurrency', '1', # Number of games played in parallel
    '-tournament', 'gauntlet'
]

print("Running Elo evaluation with cutechess-cli...")
try:
    process = subprocess.run(cutechess_command, capture_output=True, text=True, check=True)
    print("Cutechess-cli output:")
    print(process.stdout)
    if process.stderr:
        print("Cutechess-cli errors:")
        print(process.stderr)
    print(f"Results saved to {RESULTS_FILE}")
except FileNotFoundError:
    print("Error: 'cutechess-cli' or 'stockfish' not found. Please ensure they are installed and in your system's PATH.")
except subprocess.CalledProcessError as e:
    print(f"Error running cutechess-cli: {e}")
    print(f"Stdout: {e.stdout}")
    print(f"Stderr: {e.stderr}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## 5. Clean Up (Optional)
Remove the generated UCI engine script.

In [ ]:
if os.path.exists(UCI_ENGINE_SCRIPT_NAME):
    os.remove(UCI_ENGINE_SCRIPT_NAME)
    print(f"Removed '{UCI_ENGINE_SCRIPT_NAME}'.")